# AutoML Pipeline: Feature Selection + CatBoost (GPU)

Стратегия:
1. Загрузка main + extra features
2. Feature engineering: агрегат пропусков, заполнение медианой
3. Первичный отбор признаков по важности (быстрая модель)
4. Итеративный отбор: добавляем/убираем признаки по CV score
5. Финальная модель + submission

Метрика: Macro Averaged ROC-AUC

In [1]:
import polars as pl
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import gc
import warnings
warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

# Автоопределение GPU
def _check_gpu():
    try:
        model = CatBoostClassifier(task_type='GPU', iterations=1, verbose=0)
        model.fit([[0, 1], [1, 0]], [0, 1])
        return True
    except Exception:
        return False

USE_GPU = _check_gpu()
TASK_TYPE = 'GPU' if USE_GPU else 'CPU'
print(f'CatBoost device: {TASK_TYPE}')

CatBoost device: CPU


## 1. Загрузка данных

In [2]:
train_main = pl.read_parquet('train_main_features.parquet')
test_main = pl.read_parquet('test_main_features.parquet')
target = pl.read_parquet('train_target.parquet')

train_extra = pl.read_parquet('train_extra_features.parquet')
test_extra = pl.read_parquet('test_extra_features.parquet')

print(f'Main:  train={train_main.shape}, test={test_main.shape}')
print(f'Extra: train={train_extra.shape}, test={test_extra.shape}')
print(f'Target: {target.shape}')

Main:  train=(750000, 200), test=(250000, 200)
Extra: train=(750000, 2242), test=(250000, 2242)
Target: (750000, 42)


In [3]:
# Объединяем main + extra по customer_id
train_full = train_main.join(train_extra, on='customer_id', how='left')
test_full = test_main.join(test_extra, on='customer_id', how='left')

del train_main, test_main, train_extra, test_extra
gc.collect()

print(f'Full: train={train_full.shape}, test={test_full.shape}')

Full: train=(750000, 2441), test=(250000, 2441)


In [4]:
# Разделяем типы признаков
feature_cols = [c for c in train_full.columns if c != 'customer_id']
cat_cols = [c for c in feature_cols if c.startswith('cat_feature')]
num_cols = [c for c in feature_cols if c.startswith('num_feature')]
target_cols = [c for c in target.columns if c.startswith('target')]

print(f'Категориальных: {len(cat_cols)}')
print(f'Числовых: {len(num_cols)}')
print(f'Таргетов: {len(target_cols)}')

Категориальных: 67
Числовых: 2373
Таргетов: 41


## 2. Feature Engineering

In [5]:
def add_null_count_features(df, num_cols_list):
    """Общее кол-во пропусков по числовым признакам."""
    df = df.with_columns(
        pl.sum_horizontal(*[pl.col(c).is_null().cast(pl.Int16) for c in num_cols_list]).alias('total_null_count')
    )
    return df, ['total_null_count']


# Добавляем только агрегат — без сотен per-feature индикаторов
train_full, null_count_cols = add_null_count_features(train_full, num_cols)
test_full, _ = add_null_count_features(test_full, num_cols)

# Заполняем пропуски в числовых признаках медианой по train
fill_values = {}
for c in num_cols:
    median_val = train_full[c].median()
    if median_val is None:
        median_val = 0.0
    fill_values[c] = median_val

train_full = train_full.with_columns([
    pl.col(c).fill_null(v) for c, v in fill_values.items()
])
test_full = test_full.with_columns([
    pl.col(c).fill_null(v) for c, v in fill_values.items()
])

null_indicator_cols = []  # индикаторы не создаём

print(f'Добавлено агрегатов пропусков: {len(null_count_cols)}')
print(f'Числовые NaN заполнены медианой')

Добавлено агрегатов пропусков: 1
Числовые NaN заполнены медианой


In [3]:
# Обновляем список всех признаков
all_feature_cols = [c for c in train_full.columns if c != 'customer_id']
all_cat_cols = [c for c in all_feature_cols if c.startswith('cat_feature')]

print(f'Всего признаков: {len(all_feature_cols)}')
print(f'Из них категориальных: {len(all_cat_cols)}')

NameError: name 'train_full' is not defined

## 3. Удаление бесполезных признаков

Убираем признаки с >99% пропусков и почти константные.

In [2]:
n_train = len(train_full)
cols_to_drop = []

for c in all_feature_cols:
    null_pct = train_full[c].null_count() / n_train
    if null_pct > 0.99:
        cols_to_drop.append(c)
        continue
    n_unique = train_full[c].n_unique()
    if n_unique <= 1:
        cols_to_drop.append(c)

print(f'Удаляем {len(cols_to_drop)} признаков (>99% null или константные)')

if cols_to_drop:
    train_full = train_full.drop(cols_to_drop)
    test_full = test_full.drop([c for c in cols_to_drop if c in test_full.columns])

all_feature_cols = [c for c in train_full.columns if c != 'customer_id']
all_cat_cols = [c for c in all_feature_cols if c.startswith('cat_feature')]
print(f'Осталось признаков: {len(all_feature_cols)}')

NameError: name 'train_full' is not defined

## 4. Подготовка данных для CatBoost

In [1]:
# Конвертируем в pandas (float32 для экономии памяти)
num_feature_cols_set = set(all_feature_cols) - set(all_cat_cols)
num_feature_cols_list = [c for c in all_feature_cols if c in num_feature_cols_set]

train_f32 = train_full.select(all_feature_cols).with_columns([
    pl.col(c).cast(pl.Float32) for c in num_feature_cols_list
])
test_f32 = test_full.select(all_feature_cols).with_columns([
    pl.col(c).cast(pl.Float32) for c in num_feature_cols_list
])

del train_full, test_full
gc.collect()

X_train = train_f32.to_pandas()
del train_f32; gc.collect()

X_test = test_f32.to_pandas()
del test_f32; gc.collect()

Y_train = target.select(target_cols).to_pandas().values

# Категориальные: заполняем NaN и конвертируем в int (CatBoost требует int/str для cat)
cat_indices = [i for i, c in enumerate(all_feature_cols) if c in all_cat_cols]
for c in all_cat_cols:
    if c in X_train.columns:
        X_train[c] = X_train[c].fillna(-999).astype(int)
        X_test[c] = X_test[c].fillna(-999).astype(int)

# Страховка для оставшихся NaN
remaining_nan = X_train[num_feature_cols_list].isnull().sum().sum()
if remaining_nan > 0:
    print(f'WARN: осталось {remaining_nan} NaN в числовых, заполняем 0')
    X_train[num_feature_cols_list] = X_train[num_feature_cols_list].fillna(0)
    X_test[num_feature_cols_list] = X_test[num_feature_cols_list].fillna(0)

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'Y_train: {Y_train.shape}')
print(f'Cat features: {len(cat_indices)}')

NameError: name 'all_feature_cols' is not defined

## 5. Быстрая оценка важности признаков

Обучаем быструю модель на 3 самых сбалансированных таргетах и собираем feature importance.

In [ ]:
# Выбираем 3 самых сбалансированных таргета для оценки важности
target_means = Y_train.mean(axis=0)
balanced_idx = np.argsort(np.abs(target_means - 0.5))[:3]
print(f'Таргеты для оценки важности: {[target_cols[i] for i in balanced_idx]}')
print(f'Их prevalence: {[round(target_means[i], 3) for i in balanced_idx]}')

quick_params = dict(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=3,
    task_type=TASK_TYPE,
    random_seed=SEED,
    verbose=0,
    loss_function='Logloss',
    eval_metric='AUC',
    auto_class_weights='Balanced',
)

importances = np.zeros(len(all_feature_cols))

for ti in balanced_idx:
    y = Y_train[:, ti]
    model = CatBoostClassifier(**quick_params)
    model.fit(X_train, y, cat_features=cat_indices)
    importances += model.get_feature_importance()
    auc = roc_auc_score(y, model.predict_proba(X_train)[:, 1])
    print(f'  {target_cols[ti]}: train AUC = {auc:.4f}')

importances /= len(balanced_idx)

feat_imp = sorted(zip(all_feature_cols, importances), key=lambda x: -x[1])
print(f'\nТоп-20 признаков:')
for name, imp in feat_imp[:20]:
    print(f'  {name}: {imp:.2f}')

In [10]:
# Отсекаем признаки с нулевой важностью
selected_features = [name for name, imp in feat_imp if imp > 0]
zero_imp = [name for name, imp in feat_imp if imp == 0]

print(f'Признаков с importance > 0: {len(selected_features)}')
print(f'Убрано с нулевой важностью: {len(zero_imp)}')

Признаков с importance > 0: 1071
Убрано с нулевой важностью: 1212


## 6. Cross-Validation: полная оценка всех таргетов

Для каждого таргета обучаем LightGBM с CV и считаем ROC-AUC.

In [ ]:
def train_and_evaluate(X, Y, feature_subset, target_cols, cat_cols_set,
                       n_folds=5, params=None, early_stopping=50):
    """
    Обучает CatBoost для каждого таргета с CV.
    Возвращает: oof_predictions, test_predictions, per-target AUC scores.
    """
    if params is None:
        params = dict(
            iterations=1000,
            learning_rate=0.05,
            depth=8,
            l2_leaf_reg=3,
            task_type=TASK_TYPE,
            random_seed=SEED,
            verbose=0,
            loss_function='Logloss',
            eval_metric='AUC',
            auto_class_weights='Balanced',
        )

    subset_cat_idx = [i for i, c in enumerate(feature_subset) if c in cat_cols_set]

    X_sub = X[feature_subset]
    n_targets = Y.shape[1]

    oof_preds = np.zeros((len(X_sub), n_targets))
    test_preds = np.zeros((len(X_test), n_targets))
    auc_scores = []

    for ti in range(n_targets):
        y = Y[:, ti]

        n_pos = y.sum()
        if n_pos < n_folds:
            oof_preds[:, ti] = y.mean()
            test_preds[:, ti] = y.mean()
            auc_scores.append(0.5)
            print(f'  {target_cols[ti]}: SKIP (n_pos={int(n_pos)}), AUC=0.5000')
            continue

        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
        fold_aucs = []

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_sub, y)):
            X_tr, X_val = X_sub.iloc[tr_idx], X_sub.iloc[val_idx]
            y_tr, y_val = y[tr_idx], y[val_idx]

            model = CatBoostClassifier(**params, early_stopping_rounds=early_stopping)
            model.fit(
                X_tr, y_tr,
                eval_set=(X_val, y_val),
                cat_features=subset_cat_idx,
            )

            val_pred = model.predict_proba(X_val)[:, 1]
            oof_preds[val_idx, ti] = val_pred
            test_preds[:, ti] += model.predict_proba(X_test[feature_subset])[:, 1] / n_folds

            fold_auc = roc_auc_score(y_val, val_pred)
            fold_aucs.append(fold_auc)

        mean_auc = np.mean(fold_aucs)
        auc_scores.append(mean_auc)
        print(f'  {target_cols[ti]}: AUC={mean_auc:.4f} (+/- {np.std(fold_aucs):.4f})')

    macro_auc = np.mean(auc_scores)
    print(f'\n  >>> Macro AUC: {macro_auc:.4f}')

    return oof_preds, test_preds, auc_scores, macro_auc

In [ ]:
print(f'=== Baseline: все {len(selected_features)} признаков с importance > 0 ===')
oof_base, test_base, aucs_base, macro_base = train_and_evaluate(
    X_train, Y_train, selected_features, target_cols, set(all_cat_cols),
    n_folds=N_FOLDS
)

## 7. Итеративный отбор признаков

Стратегия: начинаем с топ-N признаков, постепенно добавляем группы и оцениваем прирост.

In [ ]:
def iterative_feature_selection(X, Y, ranked_features, target_cols, cat_cols_set,
                                 steps=[50, 100, 200, 400, 800], n_folds=3):
    results = []

    fast_params = dict(
        iterations=500,
        learning_rate=0.08,
        depth=6,
        l2_leaf_reg=5,
        task_type=TASK_TYPE,
        random_seed=SEED,
        verbose=0,
        loss_function='Logloss',
        eval_metric='AUC',
        auto_class_weights='Balanced',
    )

    for n_feats in steps:
        if n_feats > len(ranked_features):
            n_feats = len(ranked_features)

        subset = ranked_features[:n_feats]
        print(f'\n{"="*60}')
        print(f'Тестируем топ-{n_feats} признаков')
        print(f'{"="*60}')

        _, _, aucs, macro = train_and_evaluate(
            X, Y, subset, target_cols, cat_cols_set,
            n_folds=n_folds, params=fast_params, early_stopping=30
        )
        results.append({'n_features': n_feats, 'macro_auc': macro, 'features': subset})

        if n_feats == len(ranked_features):
            break

    return results

ranked_features = [name for name, _ in feat_imp if name in selected_features]

selection_results = iterative_feature_selection(
    X_train, Y_train, ranked_features, target_cols, set(all_cat_cols),
    steps=[50, 100, 200, 400, 800, len(selected_features)],
    n_folds=3
)

In [ ]:
# Результаты отбора
print('\nРезультаты итеративного отбора:')
print(f'{"N features":>12} | {"Macro AUC":>10}')
print('-' * 27)

best_result = max(selection_results, key=lambda x: x['macro_auc'])

for r in selection_results:
    marker = ' <-- BEST' if r == best_result else ''
    print(f'{r["n_features"]:>12} | {r["macro_auc"]:>10.4f}{marker}')

best_features = best_result['features']
print(f'\nЛучший набор: {best_result["n_features"]} признаков, Macro AUC = {best_result["macro_auc"]:.4f}')

## 8. Финальная модель

Обучаем с лучшим набором признаков и более агрессивными параметрами.

In [ ]:
final_params = dict(
    iterations=3000,
    learning_rate=0.02,
    depth=8,
    l2_leaf_reg=1,
    random_strength=0.5,
    bagging_temperature=0.8,
    task_type=TASK_TYPE,
    random_seed=SEED,
    verbose=0,
    loss_function='Logloss',
    eval_metric='AUC',
    auto_class_weights='Balanced',
)

print(f'=== Финальная модель: {len(best_features)} признаков, {N_FOLDS} фолдов ===')
oof_final, test_final, aucs_final, macro_final = train_and_evaluate(
    X_train, Y_train, best_features, target_cols, set(all_cat_cols),
    n_folds=N_FOLDS, params=final_params, early_stopping=100
)

In [ ]:
import matplotlib.pyplot as plt

# Визуализация per-target AUC
sorted_idx = np.argsort(aucs_final)
fig, ax = plt.subplots(figsize=(14, 6))
colors = ['red' if a < 0.6 else 'orange' if a < 0.7 else 'steelblue' for a in np.array(aucs_final)[sorted_idx]]
ax.bar(range(len(target_cols)), np.array(aucs_final)[sorted_idx], color=colors)
ax.axhline(macro_final, color='black', ls='--', label=f'Macro AUC = {macro_final:.4f}')
ax.set_xticks(range(len(target_cols)))
ax.set_xticklabels([target_cols[i] for i in sorted_idx], rotation=90, fontsize=7)
ax.set_ylabel('ROC-AUC')
ax.set_title('Per-target ROC-AUC (финальная модель)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Формирование submission

In [ ]:
# Берём customer_id из test
test_ids = test_full.select('customer_id')

predict_cols = [c.replace('target_', 'predict_') for c in target_cols]

submit = test_ids.hstack(
    pl.DataFrame(test_final, schema=predict_cols)
)

print(f'Submit shape: {submit.shape}')
print(submit.head())

In [ ]:
submit.write_parquet('submission_automl.parquet')
print('Submission saved: submission_automl.parquet')

## 10. Бонус: Per-target тюнинг (опционально)

Для таргетов с AUC < 0.6 пробуем другие параметры.

In [ ]:
weak_targets_idx = [i for i, a in enumerate(aucs_final) if a < 0.6]
print(f'Слабых таргетов (AUC < 0.6): {len(weak_targets_idx)}')

if weak_targets_idx:
    print('Пробуем агрессивные параметры для слабых таргетов...\n')

    aggressive_params = dict(
        iterations=5000,
        learning_rate=0.01,
        depth=10,
        l2_leaf_reg=0.5,
        random_strength=1.0,
        bagging_temperature=1.0,
        task_type=TASK_TYPE,
        random_seed=SEED,
        verbose=0,
        loss_function='Logloss',
        eval_metric='AUC',
        auto_class_weights='Balanced',
    )

    subset_cat_idx = [i for i, c in enumerate(best_features) if c in all_cat_cols]

    for ti in weak_targets_idx:
        y = Y_train[:, ti]
        n_pos = y.sum()
        if n_pos < N_FOLDS:
            continue

        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
        fold_aucs = []
        ti_test_pred = np.zeros(len(X_test))

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train[best_features], y)):
            X_tr = X_train[best_features].iloc[tr_idx]
            X_val = X_train[best_features].iloc[val_idx]
            y_tr, y_val = y[tr_idx], y[val_idx]

            model = CatBoostClassifier(**aggressive_params, early_stopping_rounds=150)
            model.fit(
                X_tr, y_tr,
                eval_set=(X_val, y_val),
                cat_features=subset_cat_idx,
            )

            val_pred = model.predict_proba(X_val)[:, 1]
            fold_aucs.append(roc_auc_score(y_val, val_pred))
            ti_test_pred += model.predict_proba(X_test[best_features])[:, 1] / N_FOLDS

        new_auc = np.mean(fold_aucs)
        old_auc = aucs_final[ti]

        if new_auc > old_auc:
            print(f'  {target_cols[ti]}: {old_auc:.4f} -> {new_auc:.4f} (IMPROVED)')
            test_final[:, ti] = ti_test_pred
            aucs_final[ti] = new_auc
        else:
            print(f'  {target_cols[ti]}: {old_auc:.4f} -> {new_auc:.4f} (keeping original)')

    submit_v2 = test_ids.hstack(pl.DataFrame(test_final, schema=predict_cols))
    submit_v2.write_parquet('submission_automl_v2.parquet')

    new_macro = np.mean(aucs_final)
    print(f'\nОбновленный Macro AUC: {new_macro:.4f}')
    print('Saved: submission_automl_v2.parquet')
else:
    print('Все таргеты имеют AUC >= 0.6, дополнительный тюнинг не требуется.')

## Итоги

Пайплайн выполнил:
1. Объединение main + extra features
2. Feature engineering (индикаторы пропусков)
3. Удаление бесполезных признаков (>99% null, константные)
4. Оценку важности признаков по 3 сбалансированным таргетам
5. Итеративный отбор: перебрал наборы из топ-50/100/200/400/800/все
6. Финальную модель с лучшим набором признаков
7. Per-target тюнинг для слабых таргетов